### prepair modules and bases settings

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GridSearchCV
from sklearn.metrics import confusion_matrix,  classification_report, log_loss
from sklearn.preprocessing import StandardScaler
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.svm import SVC
# from sklearn.ensemble import RandomForestClassifier
from scipy.stats import norm
import scipy.io
import re
import itertools

import os
from os.path import join
import contextlib
from copy import deepcopy
import imp 
import time 
import sys

import pickle
from pdb import set_trace

from IPython.display import clear_output, display

In [2]:
# Add the directory containing your modules to the Python path
sys.path.append(os.path.abspath(os.path.join('..', 'ses2_modelstims')))

# load local functions
import stim_io
import stim_io_plotting
import vtc
import bvbabel

/home/jorvhar/miniconda3/envs/predlis/lib/python3.8/site-packages/nilearn/glm/__init__.py:55: FutureWarning: The nilearn.glm module is experimental. It may change in any future release of Nilearn.
  warn('The nilearn.glm module is experimental. '


In [3]:
import regression
import varpar

In [4]:
## LOADING GRID

# Load from MAT file
variables = scipy.io.loadmat('/media/jorvhar/Data8T/MRIData/timing data/grid_parameters_python.mat')

# Extract individual variables
tunsteps = variables['tunsteps']
freqstep = variables['freqstep']
subsample = variables['subsample']
mustep = variables['mustep']
muarray_bins = variables['muarray_bins']
muarray = variables['muarray']
fwhm = variables['fwhm']
octgrid = variables['octgrid']
sigmagrid = variables['sigmagrid']
pref_range = variables['pref_range']
sharp_range_fwhm = variables['sharp_range_fwhm']
sharp_range = variables['sharp_range']

## 1. Set up regresiion model
Options:

In [5]:
### --- REGRESSION SAVING OPTIONS ---

# set modeltype
# modeltype = LinearRegression() #can be LinearRegression (ols), Ridge(alpha=..), Lasso(alpha=..)  etc.
modeltype = LinearRegression() 
key_ai = ['raw_scores', 'coefs', 'intercepts', 'correlation'] # what keys to median and mean across folds

# model return options
save_predict = False          # save y_pred-y
score_of_interest = 'score'   # what score to use  'score', 'raw_scores', 'coefs', 'intercepts', 'correlation'

# outlier options - #tobeimplemented
SD_lim = 3                    # remove y x sd higher then mean
remove_outliers = False       # if false dont remove sd outliers 

# what regressor variant to use
convolved = True   # use convolved dataset
resampled = True   # use scipy resampled data, instead of standard mean for downsampled data

zs=True  # zscore y
ts=True # temporally smooth y
hp=False  # highpass filter y

# add drift regressors
dr=False   # drift regressor

### --- LOCATION OPTIONS ---

# file location
mridat_dir = '/media/jorvhar/Data8T/MRIData/PreProc'
logdat_dir = '/media/jorvhar/Data8T/MRIData/timing data/data'
vtc_dir = '/media/jorvhar/New Volume1/vtcs' # adviced to put vtc's on a (nvme) ssd while running analyses
pp_dir = lambda pp, ses : f'S{pp:02d}_SES{ses}'
betas_dir = 'Betas'

# tonotopy and mask filenames
tonotopy_vmp = 'prf_permutations_for_s2.vmp'
mask_fn = 'gm-subcortical.msk'

# fn lambda
fn = lambda pp, ses, run : f'S{pp:02d}_SES{ses}_run{run}_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc'


### --- PARTICIPANT OPTIONS ---

# session of interest
ses = 2

# variable that may be different per participant
ppz = [1,2,3,4,5,6,7,8,9,10]
n_splitsz = [6,6,5,5,5,5,5,5,5,5]               # 12 runs > 10:2cross, 6 fold, splits used per pp (for variable length option)
n_splitsz = [12,12,10,10,10,10,10,10,10,10]               # 12 runs > 10:2cross, 6 fold, splits used per pp (for variable length option)
n_runz = [12,12,10,10,10,10,10,10,10,10]        # number of runs
startpp = 1

### --- SET THEORIE REGRESSION MODELS ---

# for full 3 sets we need 7 sets
models = ['base',
          'prediction_prior',
          'prediction_error',
          'base_U_prediction_prior',
          'base_U_prediction_error',
          'prediction_prior_U_prediction_error',
          'base_U_prediction_prior_U_prediction_error']
# if we want to use sets we only need 3 models 
##models=['base_U_adaptation', 'prediction', 'base_U_adaptation_U_prediction']

## REGRESSORS IN MODELS ##
model_regressors = {'base':       ['raw_acti', 'onoff', 'raw_adapt'], 
                    'prediction_prior': ['pred_prob', 'precision'],              #exp: deep 5/6
                    'prediction_error': ['surprisal', 'prec_w_surprisal']        #exp: superfisial 2/3 - just precion weighted?
                   } 
# if we want to add adapted activation
# model_regressors['adaptation'] += ['adapt_activ']

# set combination of regressors
model_regressors.update({'base_U_prediction_prior':               model_regressors['base']+
                                                                  model_regressors['prediction_prior'],
                        'base_U_prediction_error':                model_regressors['base']+
                                                                  model_regressors['prediction_error'], 
                        'prediction_prior_U_prediction_error':    model_regressors['prediction_prior']+
                                                                  model_regressors['prediction_error'], 
                        'base_U_prediction_prior_U_prediction_error': model_regressors['base']+
                                                                  model_regressors['prediction_prior']+
                                                                  model_regressors['prediction_error']})
y_var = 'voxeltimecourse'

In [ ]:
prior : ['pred_prob', 'precision']
error : ['error, surprisal, prec_w_surprisal', 'glob_err']

In [6]:
## error model
model_regressors_error = {'base':       ['raw_acti', 'onoff', 'raw_adapt'], 
                    'prediction_prior': ['pred_prob', 'precision'],              #exp: deep 5/6
                    'prediction_error': ['error', 'surprisal', 'prec_w_surprisal']        #exp: superfisial 2/3 - just precion weighted?
                   } 
model_regressors_error.update({'base_U_prediction_prior':               model_regressors['base']+
                                                                  model_regressors['prediction_prior'],
                        'base_U_prediction_error':                model_regressors['base']+
                                                                  model_regressors['prediction_error'], 
                        'prediction_prior_U_prediction_error':    model_regressors['prediction_prior']+
                                                                  model_regressors['prediction_error'], 
                        'base_U_prediction_prior_U_prediction_error': model_regressors['base']+
                                                                  model_regressors['prediction_prior']+
                                                                  model_regressors['prediction_error']})

## error and global error model
model_regressors_error_ge = {'base':       ['raw_acti', 'onoff', 'raw_adapt'], 
                    'prediction_prior': ['pred_prob', 'precision'],              #exp: deep 5/6
                    'prediction_error': ['error', 'surprisal', 'prec_w_surprisal', 'glob_err']        #exp: superfisial 2/3 - just precion weighted?
                   } 
model_regressors_error_ge.update({'base_U_prediction_prior':               model_regressors['base']+
                                                                  model_regressors['prediction_prior'],
                        'base_U_prediction_error':                model_regressors['base']+
                                                                  model_regressors['prediction_error'], 
                        'prediction_prior_U_prediction_error':    model_regressors['prediction_prior']+
                                                                  model_regressors['prediction_error'], 
                        'base_U_prediction_prior_U_prediction_error': model_regressors['base']+
                                                                  model_regressors['prediction_prior']+
                                                                  model_regressors['prediction_error']})


In [6]:
## run

## 2. Run regressions - per participant - per model - per gridpostion 
Run the full regressions, looping over participants, copy pasting files to a suitable ssd location, and doing the regression for the full grid.

In [ ]:
## TEMP FOR LOOKING AT PRCISION WEIGHTING ##
modelregs = [model_regressors_error, model_regressors_error_ge]
pick_fns = ['ANTS_predictors_split_prec_pw_er_lwo', 'ANTS_predictors_split_prec_pw_er_ge_lwo']

for md_idx in range(len(modelregs)):
    model_regressors = modelregs[md_idx]
    pick_fn_prefix = pick_fns[md_idx]
## -- END TEMP -- ## -PICK_FN IS COMMENTED

    # pick_fn_prefix = 'ANTS_predictors_split_prec_pw_lwo'

    # loop over all participants
    for pp_idx in np.arange(ppz.index(startpp),len(ppz)):

        ### --- PREPARE PARTICIPANT DATA ---

        # fetch current pp vars
        pp = ppz[pp_idx]
        runz = np.arange(1,n_runz[pp_idx]+1)
        n_splits = n_splitsz[pp_idx]

        print(F'--RUNNING REGRESSION LOOP FOR PP: {pp} (runs={n_runz[pp_idx]},nr_splits={n_splits})--')

        # load stim df and tr df
    #     stim_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_stim_ideal')
    #     tr_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_tr_ideal')
        stim_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_stim_v2')
        tr_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_tr_v2')

        # create full path for vmp and mask
        mskpath = join(mridat_dir, pp_dir(pp,1), mask_fn)
        vmppath = join(mridat_dir, pp_dir(pp,1), betas_dir, tonotopy_vmp)

        # load full mask and convert to indeces
        _, msk = bvbabel.msk.read_msk(mskpath)
        msk = np.where(msk)

        # load vmp image
        vmp_head, vmp_img = bvbabel.vmp.read_vmp(vmppath)

        # load list of filenames at origin, and vtc filenames
        origin_fns = [join(mridat_dir, pp_dir(pp, ses), fn(pp,ses,run)) for run in runz]
        vtc_fns = [join(vtc_dir, fn(pp,ses,run)) for run in runz]

        # copy files to ssd for efficient and fast chuck processing
        stim_io.copy_files(origin_fns, vtc_fns)

        # load tonotopy vmp
        vmp_df = stim_io.vmp_add_realsigma(vmp_img, msk, mustep[0][0]) # 1. prfMU, 2. prfMU_hz, prfS, prfO

        ### --- RUN FULL REGRESSION ---

        # run full regression for current pp
        scores = regression.run_model_grid(tr_df,stim_df,vmp_df,vtc_fns,
                                           msk, vmp_img,
                                           pref_range,sharp_range,
                                           models, model_regressors,
                                           mustep, n_splits, modeltype, key_ai,
                                           save_predict=save_predict, 
                                           convolved=convolved, resampled=resampled,
                                           zs=zs, ts=ts, hp=hp, dr=dr)
        # clean up prints - for next pp
        clear_output(wait=True)

        # save scores
        if not os.path.exists(join(mridat_dir, pp_dir(pp, ses), 'Betas')):
            os.mkdir(join(mridat_dir, pp_dir(pp, ses), 'Betas'))

        # append the pickle result naming based on cleaning steps 
        pick_fn = pick_fn_prefix
        if ts: pick_fn = f'{pick_fn}_tempsmooth'
        if hp: pick_fn = f'{pick_fn}_highpass'
        # pickle the results
        with open(join(mridat_dir, pp_dir(pp, ses), f'Betas/{pick_fn}.pickle'), 'wb') as handle:
            pickle.dump(scores, handle, protocol=pickle.HIGHEST_PROTOCOL)
        # loading of pickled results
        ###with open(join(mridat_dir, pp_dir(pp, ses), f'Betas/{pick_fn}.pickle'), 'rb') as handle:
        ###    scores = pickle.load(handle)

        # clean up files where needed for next pp
        for fp in vtc_fns:
            os.remove(fp)

--RUNNING REGRESSION LOOP FOR PP: 1 (runs=12,nr_splits=12)--
Copied /media/jorvhar/Data8T/MRIData/PreProc/S01_SES2/S01_SES2_run1_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S01_SES2_run1_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S01_SES2/S01_SES2_run2_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S01_SES2_run2_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S01_SES2/S01_SES2_run3_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S01_SES2_run3_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S01_SES2/S01_SES2_run4_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S01_SES2_run4_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S01_SES2/S01_SES2_run5_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Vol

grid: 48/2400, 
        -current chuck took: 0.42 seconds
        -estimated time elapsed: 4.67 minutes of 233.71 minutes
grid: 49/2400, 
        -current chuck took: 4.79 seconds
        -estimated time elapsed: 4.75 minutes of 232.85 minutes
grid: 50/2400, 
        -current chuck took: 9.80 seconds
        -estimated time elapsed: 4.92 minutes of 236.03 minutes
grid: 51/2400, 
        -current chuck took: 2.63 seconds
        -estimated time elapsed: 4.96 minutes of 233.46 minutes
grid: 52/2400, 
        -current chuck took: 1.04 seconds
        -estimated time elapsed: 4.98 minutes of 229.77 minutes
grid: 53/2400, 
        -current chuck took: 0.40 seconds
        -estimated time elapsed: 4.99 minutes of 225.74 minutes
grid: 54/2400, 
        -current chuck took: 2.11 seconds
        -estimated time elapsed: 5.02 minutes of 223.12 minutes
grid: 55/2400, 
        -current chuck took: 1.26 seconds
        -estimated time elapsed: 5.04 minutes of 219.98 minutes
grid: 56/2400, 
        

### Repeat but now for IdealObserver

In [ ]:
# CHECK IF WE NEED TO RUN THIS ONE AS WELL BASED ON THE THE MAIN ANALYSIS RUN

In [ ]:
# loop over all participants
for pp_idx in np.arange(ppz.index(startpp),len(ppz)):

    ### --- PREPARE PARTICIPANT DATA ---

    # fetch current pp vars
    pp = ppz[pp_idx]
    runz = np.arange(1,n_runz[pp_idx]+1)
    n_splits = n_splitsz[pp_idx]

    print(F'--RUNNING REGRESSION LOOP FOR PP: {pp} (runs={n_runz[pp_idx]},nr_splits={n_splits})--')

    # load stim df and tr df
#     stim_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_stim_ideal')
#     tr_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_tr_ideal')
    stim_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_stim_ideal')
    tr_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_tr_ideal')

    # create full path for vmp and mask
    mskpath = join(mridat_dir, pp_dir(pp,1), mask_fn)
    vmppath = join(mridat_dir, pp_dir(pp,1), betas_dir, tonotopy_vmp)

    # load full mask and convert to indeces
    _, msk = bvbabel.msk.read_msk(mskpath)
    msk = np.where(msk)

    # load vmp image
    vmp_head, vmp_img = bvbabel.vmp.read_vmp(vmppath)

    # load list of filenames at origin, and vtc filenames
    origin_fns = [join(mridat_dir, pp_dir(pp, ses), fn(pp,ses,run)) for run in runz]
    vtc_fns = [join(vtc_dir, fn(pp,ses,run)) for run in runz]

    # copy files to ssd for efficient and fast chuck processing
    stim_io.copy_files(origin_fns, vtc_fns)

    # load tonotopy vmp
    vmp_df = stim_io.vmp_add_realsigma(vmp_img, msk, mustep[0][0]) # 1. prfMU, 2. prfMU_hz, prfS, prfO

    ### --- RUN FULL REGRESSION ---

    # run full regression for current pp
    scores = regression.run_model_grid(tr_df,stim_df,vmp_df,vtc_fns,
                                       msk, vmp_img,
                                       pref_range,sharp_range,
                                       models, model_regressors,
                                       mustep, n_splits, modeltype, key_ai,
                                       save_predict=save_predict, 
                                       convolved=convolved, resampled=resampled,
                                       zs=zs, ts=ts, hp=hp, dr=dr)
    # clean up prints - for next pp
    clear_output(wait=True)

    # save scores
    if not os.path.exists(join(mridat_dir, pp_dir(pp, ses), 'Betas')):
        os.mkdir(join(mridat_dir, pp_dir(pp, ses), 'Betas'))

    # append the pickle result naming based on cleaning steps 
    pick_fn = 'ANTS_IdealObserver_predictors_split_prec_lwo'
    if ts: pick_fn = f'{pick_fn}_tempsmooth'
    if hp: pick_fn = f'{pick_fn}_highpass'
    # pickle the results
    with open(join(mridat_dir, pp_dir(pp, ses), f'Betas/{pick_fn}.pickle'), 'wb') as handle:
        pickle.dump(scores, handle, protocol=pickle.HIGHEST_PROTOCOL)
    # loading of pickled results
    ###with open(join(mridat_dir, pp_dir(pp, ses), f'Betas/{pick_fn}.pickle'), 'rb') as handle:
    ###    scores = pickle.load(handle)

    # clean up files where needed for next pp
    for fp in vtc_fns:
        os.remove(fp)



--RUNNING REGRESSION LOOP FOR PP: 1 (runs=12,nr_splits=12)--
Copied /media/jorvhar/Data8T/MRIData/PreProc/S01_SES2/S01_SES2_run1_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S01_SES2_run1_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S01_SES2/S01_SES2_run2_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S01_SES2_run2_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S01_SES2/S01_SES2_run3_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S01_SES2_run3_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S01_SES2/S01_SES2_run4_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S01_SES2_run4_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S01_SES2/S01_SES2_run5_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Vol

grid: 48/2400, 
        -current chuck took: 2.68 seconds
        -estimated time elapsed: 4.62 minutes of 230.88 minutes
grid: 49/2400, 
        -current chuck took: 5.71 seconds
        -estimated time elapsed: 4.71 minutes of 230.83 minutes
grid: 50/2400, 
        -current chuck took: 4.34 seconds
        -estimated time elapsed: 4.79 minutes of 229.69 minutes
grid: 51/2400, 
        -current chuck took: 8.25 seconds
        -estimated time elapsed: 4.92 minutes of 231.66 minutes
grid: 52/2400, 
        -current chuck took: 0.58 seconds
        -estimated time elapsed: 4.93 minutes of 227.66 minutes
grid: 53/2400, 
        -current chuck took: 0.53 seconds
        -estimated time elapsed: 4.94 minutes of 223.77 minutes
grid: 54/2400, 
        -current chuck took: 0.34 seconds
        -estimated time elapsed: 4.95 minutes of 219.87 minutes
grid: 55/2400, 
        -current chuck took: 0.60 seconds
        -estimated time elapsed: 4.96 minutes of 216.32 minutes
grid: 56/2400, 
        

grid: 115/2400, 
        -current chuck took: 1.65 seconds
        -estimated time elapsed: 9.06 minutes of 189.02 minutes
grid: 116/2400, 
        -current chuck took: 0.66 seconds
        -estimated time elapsed: 9.07 minutes of 187.62 minutes
grid: 117/2400, 
        -current chuck took: 13.56 seconds
        -estimated time elapsed: 9.29 minutes of 190.65 minutes
grid: 118/2400, 
        -current chuck took: 1.08 seconds
        -estimated time elapsed: 9.31 minutes of 189.40 minutes
grid: 119/2400, 
        -current chuck took: 13.62 seconds
        -estimated time elapsed: 9.54 minutes of 192.39 minutes
grid: 120/2400, 
        -current chuck took: 13.98 seconds
        -estimated time elapsed: 9.77 minutes of 195.45 minutes
grid: 121/2400, 
        -current chuck took: 1.08 seconds
        -estimated time elapsed: 9.79 minutes of 194.19 minutes
grid: 122/2400, 
        -current chuck took: 0.79 seconds
        -estimated time elapsed: 9.80 minutes of 192.86 minutes
grid: 123/240

grid: 182/2400, 
        -current chuck took: 1.25 seconds
        -estimated time elapsed: 13.27 minutes of 174.97 minutes
grid: 183/2400, 
        -current chuck took: 3.62 seconds
        -estimated time elapsed: 13.33 minutes of 174.81 minutes
grid: 184/2400, 
        -current chuck took: 0.34 seconds
        -estimated time elapsed: 13.33 minutes of 173.93 minutes
grid: 185/2400, 
        -current chuck took: 2.02 seconds
        -estimated time elapsed: 13.37 minutes of 173.43 minutes
grid: 186/2400, 
        -current chuck took: 3.56 seconds
        -estimated time elapsed: 13.43 minutes of 173.26 minutes
grid: 187/2400, 
        -current chuck took: 15.05 seconds
        -estimated time elapsed: 13.68 minutes of 175.56 minutes
grid: 188/2400, 
        -current chuck took: 15.98 seconds
        -estimated time elapsed: 13.95 minutes of 178.02 minutes
grid: 189/2400, 
        -current chuck took: 0.54 seconds
        -estimated time elapsed: 13.95 minutes of 177.20 minutes
grid: 

grid: 248/2400, 
        -current chuck took: 0.61 seconds
        -estimated time elapsed: 18.19 minutes of 176.07 minutes
grid: 249/2400, 
        -current chuck took: 11.61 seconds
        -estimated time elapsed: 18.39 minutes of 177.23 minutes
grid: 250/2400, 
        -current chuck took: 14.61 seconds
        -estimated time elapsed: 18.63 minutes of 178.86 minutes
grid: 251/2400, 
        -current chuck took: 5.97 seconds
        -estimated time elapsed: 18.73 minutes of 179.10 minutes
grid: 252/2400, 
        -current chuck took: 0.96 seconds
        -estimated time elapsed: 18.75 minutes of 178.54 minutes
grid: 253/2400, 
        -current chuck took: 0.42 seconds
        -estimated time elapsed: 18.75 minutes of 177.90 minutes
grid: 254/2400, 
        -current chuck took: 8.40 seconds
        -estimated time elapsed: 18.89 minutes of 178.52 minutes
grid: 255/2400, 
        -current chuck took: 0.32 seconds
        -estimated time elapsed: 18.90 minutes of 177.87 minutes
grid: 

...

grid: 446/2400, 
        -current chuck took: 0.96 seconds
        -estimated time elapsed: 36.13 minutes of 194.44 minutes
grid: 447/2400, 
        -current chuck took: 6.24 seconds
        -estimated time elapsed: 36.24 minutes of 194.57 minutes
grid: 448/2400, 
        -current chuck took: 4.99 seconds
        -estimated time elapsed: 36.32 minutes of 194.58 minutes
grid: 449/2400, 
        -current chuck took: 5.33 seconds
        -estimated time elapsed: 36.41 minutes of 194.62 minutes
grid: 450/2400, 
        -current chuck took: 11.34 seconds
        -estimated time elapsed: 36.60 minutes of 195.19 minutes
grid: 451/2400, 
        -current chuck took: 11.90 seconds
        -estimated time elapsed: 36.80 minutes of 195.82 minutes
grid: 452/2400, 
        -current chuck took: 3.54 seconds
        -estimated time elapsed: 36.86 minutes of 195.70 minutes
grid: 453/2400, 
        -current chuck took: 5.37 seconds
        -estimated time elapsed: 36.95 minutes of 195.74 minutes
grid: 

grid: 513/2400, 
        -current chuck took: 0.48 seconds
        -estimated time elapsed: 40.51 minutes of 189.53 minutes
grid: 514/2400, 
        -current chuck took: 1.58 seconds
        -estimated time elapsed: 40.54 minutes of 189.29 minutes
grid: 515/2400, 
        -current chuck took: 0.47 seconds
        -estimated time elapsed: 40.55 minutes of 188.96 minutes
grid: 516/2400, 
        -current chuck took: 0.54 seconds
        -estimated time elapsed: 40.56 minutes of 188.63 minutes
grid: 517/2400, 
        -current chuck took: 1.48 seconds
        -estimated time elapsed: 40.58 minutes of 188.38 minutes
grid: 518/2400, 
        -current chuck took: 13.00 seconds
        -estimated time elapsed: 40.80 minutes of 189.02 minutes
grid: 519/2400, 
        -current chuck took: 6.89 seconds
        -estimated time elapsed: 40.91 minutes of 189.19 minutes
grid: 520/2400, 
        -current chuck took: 15.61 seconds
        -estimated time elapsed: 41.17 minutes of 190.03 minutes
grid: 

grid: 579/2400, 
        -current chuck took: 12.14 seconds
        -estimated time elapsed: 45.36 minutes of 188.01 minutes
grid: 580/2400, 
        -current chuck took: 9.13 seconds
        -estimated time elapsed: 45.51 minutes of 188.32 minutes
grid: 581/2400, 
        -current chuck took: 6.32 seconds
        -estimated time elapsed: 45.62 minutes of 188.43 minutes
grid: 582/2400, 
        -current chuck took: 4.20 seconds
        -estimated time elapsed: 45.69 minutes of 188.39 minutes
grid: 583/2400, 
        -current chuck took: 0.70 seconds
        -estimated time elapsed: 45.70 minutes of 188.12 minutes
grid: 584/2400, 
        -current chuck took: 1.08 seconds
        -estimated time elapsed: 45.72 minutes of 187.87 minutes
grid: 585/2400, 
        -current chuck took: 0.49 seconds
        -estimated time elapsed: 45.72 minutes of 187.58 minutes
grid: 586/2400, 
        -current chuck took: 0.48 seconds
        -estimated time elapsed: 45.73 minutes of 187.30 minutes
grid: 5

grid: 645/2400, 
        -current chuck took: 0.60 seconds
        -estimated time elapsed: 50.64 minutes of 188.45 minutes
grid: 646/2400, 
        -current chuck took: 4.50 seconds
        -estimated time elapsed: 50.72 minutes of 188.43 minutes
grid: 647/2400, 
        -current chuck took: 0.49 seconds
        -estimated time elapsed: 50.73 minutes of 188.17 minutes
grid: 648/2400, 
        -current chuck took: 13.03 seconds
        -estimated time elapsed: 50.95 minutes of 188.69 minutes
grid: 649/2400, 
        -current chuck took: 11.77 seconds
        -estimated time elapsed: 51.14 minutes of 189.12 minutes
grid: 650/2400, 
        -current chuck took: 4.83 seconds
        -estimated time elapsed: 51.22 minutes of 189.13 minutes
grid: 651/2400, 
        -current chuck took: 6.37 seconds
        -estimated time elapsed: 51.33 minutes of 189.23 minutes
grid: 652/2400, 
        -current chuck took: 6.56 seconds
        -estimated time elapsed: 51.44 minutes of 189.34 minutes
grid: 

grid: 711/2400, 
        -current chuck took: 0.86 seconds
        -estimated time elapsed: 55.32 minutes of 186.72 minutes
grid: 712/2400, 
        -current chuck took: 0.66 seconds
        -estimated time elapsed: 55.33 minutes of 186.49 minutes
grid: 713/2400, 
        -current chuck took: 1.98 seconds
        -estimated time elapsed: 55.36 minutes of 186.34 minutes
grid: 714/2400, 
        -current chuck took: 0.51 seconds
        -estimated time elapsed: 55.37 minutes of 186.11 minutes
grid: 715/2400, 
        -current chuck took: 3.00 seconds
        -estimated time elapsed: 55.42 minutes of 186.02 minutes
grid: 716/2400, 
        -current chuck took: 0.50 seconds
        -estimated time elapsed: 55.43 minutes of 185.79 minutes
grid: 717/2400, 
        -current chuck took: 5.54 seconds
        -estimated time elapsed: 55.52 minutes of 185.84 minutes
grid: 718/2400, 
        -current chuck took: 1.35 seconds
        -estimated time elapsed: 55.54 minutes of 185.65 minutes
grid: 71